In [1]:
import pandas as pd
import polars as pl
import numpy as np
import os
import re
from collections import Counter
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from sklearn.preprocessing import MinMaxScaler


In [2]:
nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package punkt to /Users/isabel/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/isabel/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [3]:
INPUT_DIR = 'fromGoogleDrive'
OUTPUT_DIR = 'results'

In [4]:
os.makedirs(f'./{OUTPUT_DIR}/', exist_ok = True)

In [5]:
# preprocessing (same as compcor)
def preprocess(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^\s*-\s*", "", text) # Remove dashes at the beginning of texts.
    text = re.sub(r"^\s*\d+\.\s*", "", text) # Remove numbers in 1., 2., 3. format at the beginning of the text.
    words = word_tokenize(text) # tokenize
    return words

In [6]:
metric_cols = [
    "Accuracy",
    "Weighted Accuracy",
    "Time",
    "Monotonicity",
    "Separability",
    "Linearity"
]

In [7]:
# Get dataset-specific information.
rows = []
for file in os.listdir(f'./{INPUT_DIR}/datasets/datasetsPrep/'):
    # if 'dementia' not in file:

        # Process texts.
        list_of_texts = [preprocess(t) for t in pd.read_csv(f'./{INPUT_DIR}/datasets/datasetsPrep/{file}')['text'].dropna().tolist()]
        # Flatten words.
        all_words = [word for doc in list_of_texts for word in doc]
        total_words = len(all_words)
        freq = Counter(all_words)
        # Filter stopwords.
        filtered_words = [w for w in all_words if w.lower() not in stop_words]
        filtered_word_total = len(filtered_words)
        stopword_ratio = (total_words - filtered_word_total) / total_words if total_words > 0 else 0
        # Average word length.
        avg_word_len = np.mean([len(w) for w in all_words]) if all_words else 0
        # Get document lengths.
        doc_lengths = np.array([len(t) for t in list_of_texts])
        # Vocab.
        vocab = set(all_words)
        vocab_len = len(vocab)
        # Type-Token Ratio.
        ttr = vocab_len / total_words if total_words > 0 else 0
        # Rare words (hapax legomena).
        hapax = sum(1 for _, c in freq.items() if c == 1)

        # Rare words (dis legomena).
        dis = sum(1 for _, c in freq.items() if c == 2)

        # Top-N Coverage (Frequency Concentration)
        top_10_count = sum(c for _, c in freq.most_common(10))
        coverage = top_10_count / total_words if total_words > 0 else 0
        # Append everything to a row. 
        rows.append({
        "Dataset": file.replace('.csv', ''),
        "Total Documents": len(list_of_texts),
        "Total Words": total_words,
        "Total Words (without stopwords)": filtered_word_total,
        "Total Stop Words": total_words - filtered_word_total,
        "Stopword Ratio": stopword_ratio,
        "Average Word Length": avg_word_len,
        "Vocab Length": vocab_len,
        "Type-Token Ratio": ttr,
        "Hapax Legomena": hapax,
        "Dis Legomena": dis,
        "Top 10 Words Coverage": coverage,
        "Mean Document Length": doc_lengths.mean(),
        "Median Document Length": np.median(doc_lengths),
        "Standard Deviation Document Length": doc_lengths.std(),
        "Min Document Length": doc_lengths.min(),
        "Max Document Length": doc_lengths.max(),
        "25th Document Length Percentile": np.percentile(doc_lengths, 25),
        "75th Document Length Percentile": np.percentile(doc_lengths, 75),
        })

# Make a dataframe.
dataset_characteristics_df = pd.DataFrame(rows)
# Remove non-comparable metrics (e.g. number of documents) for easy visualization. 
dataset_characteristics_df.drop(columns=[ 'Total Words', 'Total Words (without stopwords)', 'Total Stop Words', 'Vocab Length', 'Hapax Legomena', 'Dis Legomena', 'Standard Deviation Document Length', 'Min Document Length', 'Max Document Length', '25th Document Length Percentile', '75th Document Length Percentile'])

,Dataset,Total Documents,Stopword Ratio,Average Word Length,Type-Token Ratio,Top 10 Words Coverage,Mean Document Length,Median Document Length
0,atis,4978,0.425348,4.707788,0.015657,0.352524,11.367818,11.0
1,banking77,13069,0.498065,3.584884,0.017522,0.310065,13.249216,11.0
2,clinc150,23700,0.498641,3.826311,0.036851,0.283618,8.525612,8.0
3,clinicalDialogueSummarizations,3603,0.398338,4.360843,0.035360,0.267681,46.278934,17.0
4,dementiaAudio,549,0.453092,3.480664,0.026194,0.386441,116.338798,105.0
5,huffPostNews,189815,0.390427,4.199670,0.023536,0.238591,25.163032,23.0
6,medicalAbstracts,14438,0.308262,5.109807,0.021201,0.263213,205.660064,200.0
7,simSUM,10000,0.124588,4.106722,0.012611,0.381137,104.821400,103.0
8,syntheticCareHomeNurseNotes,5783,0.340652,4.937623,0.027334,0.277734,26.532596,24.0
9,yahoo,87362,0.424969,3.922505,0.029172,0.255604,47.845493,42.0


In [8]:
def make_non_normalized_dfs(input_folder, output_file_name):
    all_temp_dfs = []
    for combination in os.listdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}'):
        if os.path.isdir(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}'):
            combination_splits = combination.split('_')
            dataset1, dataset2, repetitions = combination_splits[0], combination_splits[1], combination_splits[3]
            temp_df = pd.read_csv(f'./{INPUT_DIR}/outputCompcor/{input_folder}/{combination}/{combination}_ksc_metrics_measures.csv')
            grouped_df = temp_df.groupby('metric')[metric_cols].mean()
            grouped_df['Dataset 1'] = dataset1
            grouped_df['Dataset 2'] = dataset2
            grouped_df['Repetitions'] = repetitions
            grouped_df['Metric'] = grouped_df.index
            grouped_df.reset_index(inplace=True)
            grouped_df.drop(columns='metric', inplace=True)
            all_temp_dfs.append(grouped_df)

    all_dfs = pd.concat(all_temp_dfs)

    print(# All datasets should have been compared the same number of times for this section to work.
    Counter(list(all_dfs['Dataset 1']) + list(all_dfs['Dataset 2'])))

    temp_dataset_dfs = []
    for dataset in dataset_characteristics_df['Dataset']:
        temp_df = all_dfs[
                (all_dfs['Dataset 1'] == dataset) | 
                (all_dfs['Dataset 2'] == dataset)
            ].copy()
        temp_df = temp_df.groupby('Metric')[metric_cols].mean()
        temp_df['Dataset'] = dataset
        temp_dataset_dfs.append(temp_df)
    dataset_df = pd.concat(temp_dataset_dfs)
    dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}.xlsx')

    mean_dataset_df = dataset_df.groupby('Metric')[metric_cols].mean()
    mean_dataset_df.to_excel(f'./{OUTPUT_DIR}/{output_file_name}Mean.xlsx')

    return dataset_df, mean_dataset_df

In [9]:
ksc_dataset_df, ksc_mean_dataset_df = make_non_normalized_dfs('ksc', 'kscDataset')
ksc_synth_dataset_df, ksc_synth_mean_dataset_df = make_non_normalized_dfs('ksc_synth', 'kscSynthDataset')

Counter({'atis': 3472, 'banking77': 3472, 'clinc150': 3472, 'clinicalDialogueSummarizations': 3472, 'huffPostNews': 3472, 'medicalAbstracts': 3472, 'simSUM': 3472, 'syntheticCareHomeNurseNotes': 3472, 'yahoo': 3472})
Counter({'atis': 868, 'banking77': 868, 'clinc150': 868, 'clinicalDialogueSummarizations': 868, 'huffPostNews': 868, 'medicalAbstracts': 868, 'simSUM': 868, 'syntheticCareHomeNurseNotes': 868, 'yahoo': 868})


In [10]:
ksc_dataset_df['Type'] = 'KSC'
ksc_synth_dataset_df['Type'] = 'KSC_Synth'

In [11]:
all_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Dataset', 'Metric'])[metric_cols].mean()
all_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Metric'])[metric_cols].mean()
all_type_mean_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Metric'])[metric_cols].mean()
all_type_dataset_df = pd.concat([ksc_dataset_df, ksc_synth_dataset_df]).groupby(['Type', 'Dataset', 'Metric'])[metric_cols].mean()

In [12]:
all_dataset_df.to_excel(f'./{OUTPUT_DIR}/allDataset.xlsx')
all_mean_df.to_excel(f'./{OUTPUT_DIR}/allDatasetMean.xlsx')

In [13]:
all_type_mean_df

Accuracy  \
Type      Metric                                                         
KSC       cross-encoder/nli-deberta-v3-small__prompt1         0.731143   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.747129   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.743872   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.747613   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.741905   
...                                                                ...   
KSC_Synth valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.647286   
          valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.656607   
          valhalla/distilbart-mnli-12-3__prompt4              0.606889   
          valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.646993   
          valhalla/distilbart-mnli-12-3__prompt5              0.660113   

                                                              Weighted Accuracy  \
Type      Metric                                                                  
KSC       cross-encoder/nli-deberta-v3-small__prompt1                  0.687830   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.705515   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.702007   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.706729   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.698738   
...                                                                         ...   
KSC_Synth valhalla/distilbart-mnli-12-3__prompt3_prompt4_...           0.617904   
          valhalla/distilbart-mnli-12-3__prompt3_prompt5               0.624506   
          valhalla/distilbart-mnli-12-3__prompt4                       0.579888   
          valhalla/distilbart-mnli-12-3__prompt4_prompt5               0.617377   
          valhalla/distilbart-mnli-12-3__prompt5                       0.632242   

                                                                  Time  \
Type      Metric                                                         
KSC       cross-encoder/nli-deberta-v3-small__prompt1         0.073119   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.037804   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.025731   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.019302   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.015533   
...                                                                ...   
KSC_Synth valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.014189   
          valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.021527   
          valhalla/distilbart-mnli-12-3__prompt4              0.040915   
          valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.021021   
          valhalla/distilbart-mnli-12-3__prompt5              0.042489   

                                                              Monotonicity  \
Type      Metric                                                             
KSC       cross-encoder/nli-deberta-v3-small__prompt1             0.576176   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.605309   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.600836   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.603790   
          cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.597799   
...                                                                    ...   
KSC_Synth valhalla/distilbart-mnli-12-3__prompt3_prompt4_...      0.378415   
          valhalla/distilbart-mnli-12-3__prompt3_prompt5          0.397803   
          valhalla/distilbart-mnli-12-3__prompt4                  0.293162   
          valhalla/distilbart-mnli-12-3__prompt4_prompt5          0.374750   
          valhalla/distilbart-mnli-12-3__prompt5                  0.388984   

                                         

In [14]:
all_type_dataset_df

Accuracy  \
Type      Dataset Metric                                                         
KSC       atis    cross-encoder/nli-deberta-v3-small__prompt1         0.752164   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.783903   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.765299   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.770363   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.756252   
...                                                                        ...   
KSC_Synth yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.572758   
                  valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.570600   
                  valhalla/distilbart-mnli-12-3__prompt4              0.535278   
                  valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.551293   
                  valhalla/distilbart-mnli-12-3__prompt5              0.563885   

                                                                      Weighted Accuracy  \
Type      Dataset Metric                                                                  
KSC       atis    cross-encoder/nli-deberta-v3-small__prompt1                  0.702456   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.739277   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.717972   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.728748   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.712460   
...                                                                                 ...   
KSC_Synth yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...           0.550404   
                  valhalla/distilbart-mnli-12-3__prompt3_prompt5               0.548521   
                  valhalla/distilbart-mnli-12-3__prompt4                       0.523509   
                  valhalla/distilbart-mnli-12-3__prompt4_prompt5               0.527209   
                  valhalla/distilbart-mnli-12-3__prompt5                       0.552386   

                                                                          Time  \
Type      Dataset Metric                                                         
KSC       atis    cross-encoder/nli-deberta-v3-small__prompt1         0.096390   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.050168   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.034242   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.025700   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.020698   
...                                                                        ...   
KSC_Synth yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.013046   
                  valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.019800   
                  valhalla/distilbart-mnli-12-3__prompt4              0.037714   
                  valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.019339   
                  valhalla/distilbart-mnli-12-3__prompt5              0.039164   

                                                                      Monotonicity  \
Type      Dataset Metric                                                             
KSC       atis    cross-encoder/nli-deberta-v3-small__prompt1             0.642644   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.683081   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.661497   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.662021   
                  cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.634733   
...                                                                            ...   
KSC_Synth yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4

In [15]:
all_dataset_df

Accuracy  \
Dataset Metric                                                         
atis    cross-encoder/nli-deberta-v3-small__prompt1         0.740479   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.759993   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.747790   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.742790   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.723408   
...                                                              ...   
yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.631881   
        valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.632008   
        valhalla/distilbart-mnli-12-3__prompt4              0.604127   
        valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.617179   
        valhalla/distilbart-mnli-12-3__prompt5              0.630626   

                                                            Weighted Accuracy  \
Dataset Metric                                                                  
atis    cross-encoder/nli-deberta-v3-small__prompt1                  0.691584   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.714374   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.705391   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.704850   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...           0.682824   
...                                                                       ...   
yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...           0.606097   
        valhalla/distilbart-mnli-12-3__prompt3_prompt5               0.606276   
        valhalla/distilbart-mnli-12-3__prompt4                       0.579432   
        valhalla/distilbart-mnli-12-3__prompt4_prompt5               0.591193   
        valhalla/distilbart-mnli-12-3__prompt5                       0.609144   

                                                                Time  \
Dataset Metric                                                         
atis    cross-encoder/nli-deberta-v3-small__prompt1         0.109759   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.057350   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.039138   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.029384   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...  0.023662   
...                                                              ...   
yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...  0.012166   
        valhalla/distilbart-mnli-12-3__prompt3_prompt5      0.018418   
        valhalla/distilbart-mnli-12-3__prompt4              0.035359   
        valhalla/distilbart-mnli-12-3__prompt4_prompt5      0.018061   
        valhalla/distilbart-mnli-12-3__prompt5              0.036433   

                                                            Monotonicity  \
Dataset Metric                                                             
atis    cross-encoder/nli-deberta-v3-small__prompt1             0.594633   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.611402   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.606403   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.589419   
        cross-encoder/nli-deberta-v3-small__prompt1_pro...      0.568958   
...                                                                  ...   
yahoo   valhalla/distilbart-mnli-12-3__prompt3_prompt4_...      0.357312   
        valhalla/distilbart-mnli-12-3__prompt3_prompt5          0.361410   
        valhalla/distilbart-mnli-12-3__prompt4                  0.303048   
        valhalla/distilbart-mnli-12-3__prompt4_prompt5          0.346598   
        valhalla/distilbart-mnli-12-3__prompt5                  0.347464   

                                                            Separability  \
Dataset Metric                                                     

In [ ]:
all_mean_df

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity
Metric,,,,,,
cross-encoder/nli-deberta-v3-small__prompt1,0.679070,0.643832,0.080788,0.445993,0.258709,0.475663
cross-encoder/nli-deberta-v3-small__prompt1_prompt2,0.696193,0.659999,0.041887,0.477468,0.292844,0.506763
cross-encoder/nli-deberta-v3-small__prompt1_prompt2_prompt3,0.688132,0.652205,0.028535,0.466322,0.283196,0.495381
cross-encoder/nli-deberta-v3-small__prompt1_prompt2_prompt3_prompt4,0.692013,0.656954,0.021411,0.473203,0.285724,0.500322
cross-encoder/nli-deberta-v3-small__prompt1_prompt2_prompt3_prompt4_prompt5,0.680160,0.643357,0.017236,0.460765,0.278558,0.487636
...,...,...,...,...,...,...
valhalla/distilbart-mnli-12-3__prompt3_prompt4_prompt5,0.680681,0.646230,0.013075,0.458416,0.279593,0.486521
valhalla/distilbart-mnli-12-3__prompt3_prompt5,0.689590,0.653003,0.019818,0.475632,0.298889,0.505384
valhalla/distilbart-mnli-12-3__prompt4,0.650189,0.617046,0.037792,0.392246,0.204219,0.420331


In [43]:
new_rows = []
for row in all_mean_df.iterrows():
    temp_name = row[0]
    temp_row = row[1]
    split_name = temp_name.split('__')
    # single model comparisons
    if split_name[1] == 'prompt1_prompt2':
        if len(split_name[0].split('_')) == 1 or len(split_name[0].split('_')) == 3:
            temp_row['test'] = temp_name
            new_rows.append(temp_row)
    # single prompt comparisons
    if split_name[0] == 'cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3':
        if len(split_name[0].split('_')) == 3 and (len(split_name[1].split('_')) == 1 or split_name[1] == 'prompt1_prompt2'):
            temp_row['test'] = temp_name
            new_rows.append(temp_row)

final_all_mean_df = pd.DataFrame(new_rows)

In [44]:
final_all_mean_df

,Accuracy,Weighted Accuracy,Time,Monotonicity,Separability,Linearity,test
cross-encoder/nli-deberta-v3-small__prompt1_prompt2,0.696193,0.659999,0.041887,0.477468,0.292844,0.506763,cross-encoder/nli-deberta-v3-small__prompt1_pr...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt1,0.697893,0.661705,0.021160,0.499704,0.307796,0.528712,cross-encoder/nli-deberta-v3-small_typeform/di...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt1_prompt2,0.705794,0.670541,0.010566,0.513536,0.330900,0.545245,cross-encoder/nli-deberta-v3-small_typeform/di...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt1_prompt2,0.705794,0.670541,0.010566,0.513536,0.330900,0.545245,cross-encoder/nli-deberta-v3-small_typeform/di...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt2,0.699155,0.661562,0.020894,0.507790,0.332026,0.540113,cross-encoder/nli-deberta-v3-small_typeform/di...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt3,0.666837,0.633133,0.021937,0.437683,0.265076,0.471728,cross-encoder/nli-deberta-v3-small_typeform/di...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt4,0.663305,0.627531,0.021011,0.431846,0.237201,0.456303,cross-encoder/nli-deberta-v3-small_typeform/di...
cross-encoder/nli-deberta-v3-small_typeform/distilbert-base-uncased-mnli_valhalla/distilbart-mnli-12-3__prompt5,0.689172,0.652333,0.021685,0.486796,0.294233,0.520699,cross-encoder/nli-deberta-v3-small_typeform/di...
typeform/distilbert-base-uncased-mnli__prompt1_prompt2,0.670801,0.634501,0.051853,0.443422,0.248000,0.474899,typeform/distilbert-base-uncased-mnli__prompt1...
valhalla/distilbart-mnli-12-3__prompt1_prompt2,0.711160,0.675513,0.019190,0.507632,0.319851,0.531423,valhalla/distilbart-mnli-12-3__prompt1_prompt2
